In [ ]:
import ee
import datetime

# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

In [ ]:
# Indonesia bounding box
bbox = ee.Geometry.BBox(95.2930261576, -10.3599874813, 141.03385176, 5.47982086834)

# CHIRPS dataset
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterBounds(bbox)

print(f"Total CHIRPS images available: {chirps.size().getInfo()}")
print(f"Date range: {chirps.first().date().format('YYYY-MM-dd').getInfo()} to {chirps.sort('system:time_start', False).first().date().format('YYYY-MM-dd').getInfo()}")

In [ ]:
# Range of years to download
START_YEAR = 1990  # Change this to resume from specific year
END_YEAR = 2026   # Exclusive (will download up to 2024)

years = list(range(START_YEAR, END_YEAR))
print(f"Will export {len(years)} years: {years[0]}-{years[-1]}")

In [ ]:
# Export loop
for year in years:
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')
    
    # Filter to that year's daily images
    year_coll = chirps.filterDate(start, end).select('precipitation')
    
    # Check how many days in this year
    num_days = year_coll.size().getInfo()
    print(f"Year {year}: {num_days} days")
    
    # Merge all days into a single multi-band image
    year_img = year_coll.toBands()
    
    # Add time metadata
    year_img = year_img.set('system:time_start', start.millis())
    
    # Export to Drive with explicit NoData handling
    task = ee.batch.Export.image.toDrive(
        image=year_img.clip(bbox),
        description=f"CHIRPS_Daily_{year}",
        folder="Indonesia_CHIRPS_Daily_Years",
        fileNamePrefix=f"CHIRPS_Daily_{year}",
        region=bbox,
        scale=5000,  # ~5km resolution (native CHIRPS)
        maxPixels=1e13,
        formatOptions={
            'cloudOptimized': True,
            'noData': -9999.0  # ← CRITICAL FIX: Explicit NoData value
        }
    )
    task.start()
    print(f"  ✓ Exporting CHIRPS_Daily_{year} (Task ID: {task.id})")

print(f"\n{'='*70}")
print(f"All {len(years)} export tasks started!")
print(f"Monitor progress at: https://code.earthengine.google.com/tasks")
print(f"Output folder: Indonesia_CHIRPS_Daily_Years/")
print(f"{'='*70}")

In [ ]:
# Optional: Check task status
tasks = ee.batch.Task.list()
chirps_tasks = [t for t in tasks if 'CHIRPS_Daily' in t.config['description']]

print(f"\nCHIRPS Task Status:")
print(f"-" * 50)
for task in chirps_tasks[:10]:  # Show first 10
    print(f"{task.config['description']:25} {task.state}")